# MOM6 velocity truncations: where and how often

MOM6 truncates a velocity when its CFL number exceeds 0.5 (`CFL_BASED_TRUNCATIONS`), writes a record of each one to
`U_velocity_truncations` / `V_velocity_truncations`, and aborts a run after `MAXTRUNC` (10,000 by default)
truncations. This notebook reads those files and each `outputNNN/ocean.stats` from the experiment's archive folder,
then maps where truncations happen over the model bathymetry and tabulates how often.

See the [MOM6 velocity truncations](https://access-om3-configs.access-hive.org.au/latest/Tips-and-tricks/#mom6-velocity-truncations)
tips for how to read a record.

This notebook started as
[Examine_truncation_data.ipynb](https://github.com/ACCESS-NRI/access-eval-recipes/blob/ea7a8e2a97b5fe0275b2910cfe6fa089857d1b85/ocean/Examine_truncation_data.ipynb)
in access-eval-recipes (Minghang Li). Its adaptation to run unattended with `mkfigs` was drafted with Claude.

In [ ]:
# These first two cells must be in all notebooks!
# It allows us to run all the notebooks at once, this cell has a tag "parameters" which allows us to pass in
# arguments externally using papermill (see mkfigs.sh for details)

# Set esm_file to the datastore for the main experiment of interest
esm_file = "/g/data/ol01/outputs/access-om3-25km/MC_25km_jra_iaf+wombatlite-test4-d28e0359/datastore.json"

# papermill settings. *No need to modify these if running interactively.*
papermill = False                      # `cwd` and `nbname` will be populated by papermill.
cwd = None                             # current working directory
nbname = None                          # notebook name


In [ ]:
if not papermill:
    import nci_ipynb, os  # requires conda/analysis3-26.03 or later
    cwd = nci_ipynb.dir()
    nbname = nci_ipynb.name()
    os.chdir(cwd)
import mkfigs_bootstrap  # noqa: adds external/access-model-mkfigs/src to sys.path (stop-gap)
from mkfigs import MkmdWriter
mkmd = MkmdWriter(esm_file, nbname, str(cwd), pm=papermill)


In [ ]:
import glob
import os
import re
from collections import Counter

import intake
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr


### Find the archive folder

The archive folder is taken from the paths of the files in the datastore (the parent of the `outputNNN` folders),
not from where `esm_file` lives: the nightly monitor builds a temporary datastore elsewhere.

In [ ]:
COLUMNS_WITH_ITERABLES = ["variable", "variable_long_name", "variable_standard_name",
                          "variable_cell_methods", "variable_units"]
datastore = intake.open_esm_datastore(esm_file, columns_with_iterables=COLUMNS_WITH_ITERABLES)

archives = Counter(m.group(1) for p in datastore.df["path"] if (m := re.match(r"(.*)/output\d+/", p)))
archive = archives.most_common(1)[0][0] if archives else os.path.dirname(esm_file)
outputs = sorted(glob.glob(os.path.join(archive, "output[0-9]*")))
print(f"{archive}: {len(outputs)} output folders")


### Truncations per output folder (`ocean.stats`)

`ocean.stats` counts every truncation (its Truncs column), whereas the truncation files hold at most 50 records per
PE per run (`MAX_TRUNC_FILE_SIZE_PER_PE`), so totals come from `ocean.stats`.

In [ ]:
def read_ocean_stats(path):
    """(total truncations, max CFL) from one ocean.stats file."""
    truncs, cfl = 0, 0.0
    with open(path, errors="replace") as f:
        for line in f:
            fields = line.split(",")
            if len(fields) > 4 and fields[0].strip().isdigit():
                truncs += int(float(fields[2]))
                cfl = max(cfl, float(fields[4].split()[1]))
    return truncs, cfl


per_output = pd.DataFrame(
    [(os.path.basename(o), *read_ocean_stats(os.path.join(o, "ocean.stats")))
     for o in outputs if os.path.isfile(os.path.join(o, "ocean.stats"))],
    columns=["output", "truncations", "max_cfl"],
)
per_output


### Truncation records (`U_velocity_truncations`, `V_velocity_truncations`)

Each record starts with a header line such as

`Time 1900 0   1.97 V-velocity violation at 6051: 36, 14 (   2.90 E   79.16 N) Layers 38 to 42. dt =  240.0`

(older files write the indices without a comma: `at 2009: 21 5`). The velocity, CFL and depth come from the
`u(m)`/`v(m)`, `CFL u`/`CFL v` and `D` lines that follow.

In [ ]:
HEADER = re.compile(
    r"Time\s+(\d+)\s+(\d+)\s+([\d.]+)\s+([UV])-velocity violation at\s+(\d+):\s*(\d+),?\s+(\d+)\s*"
    r"\(\s*(-?[\d.]+)\s*E\s+(-?[\d.]+)\s*N\)\s*Layers\s+(\d+)\s+to\s+(\d+)\.\s*dt\s*=\s*([\d.]+)")


def numbers(line):
    """The numbers after the first ':' on a record line."""
    out = []
    for tok in line.split(":", 1)[1].split():
        try:
            out.append(float(tok))
        except ValueError:
            pass
    return out


def read_truncations(path, output):
    """One dict per record in a U/V_velocity_truncations file."""
    records, rec = [], None
    with open(path, errors="replace") as f:
        for line in f:
            m = HEADER.search(line)
            if m:
                rec = dict(output=output, year=int(m[1]), yearday=int(m[2]), uv=m[4], pe=int(m[5]),
                           lon=float(m[8]), lat=float(m[9]), k0=int(m[10]), k1=int(m[11]), dt=float(m[12]),
                           speed=np.nan, cfl=np.nan, depth=np.nan)
                records.append(rec)
            elif rec and ":" in line:
                name, vals = line.split(":", 1)[0].strip(), numbers(line)
                if vals and name in ("u(m)", "v(m)"):
                    rec["speed"] = max(abs(v) for v in vals)
                elif vals and name in ("CFL u", "CFL v"):
                    rec["cfl"] = max(vals)
                elif vals and name == "D":
                    rec["depth"] = min(vals)
    return records


rows = []
for o in outputs:
    for uv in "UV":
        path = os.path.join(o, f"{uv}_velocity_truncations")
        if os.path.isfile(path):
            rows += read_truncations(path, os.path.basename(o))
records = pd.DataFrame(rows, columns=["output", "year", "yearday", "uv", "pe", "lon", "lat", "k0", "k1", "dt",
                                      "speed", "cfl", "depth"])
# PEs that wrote the maximum 50 records in a run: their true counts are higher
capped = set(records.groupby(["output", "pe"]).size().loc[lambda n: n >= 50].index)
records["at_cap"] = pd.MultiIndex.from_frame(records[["output", "pe"]]).isin(capped)
capped_pes = {pe for _, pe in capped}
print(f"{len(records)} records; PEs at the 50-record cap: {sorted(capped_pes) or 'none'}")

### Summary

In [ ]:
total = int(per_output["truncations"].sum()) if len(per_output) else 0
if len(per_output) == 0:
    mkmd.table("MOM6 velocity truncations", ["| |", "|---|", "| No `ocean.stats` files found in the output folders. |"])
elif total == 0:
    mkmd.table("MOM6 velocity truncations",
               ["| |", "|---|", f"| No velocity truncations: the Truncs column of `ocean.stats` is zero in all {len(per_output)} output folders. |"])
else:
    rows = ["| Output folder | Truncations (ocean.stats) | Max CFL | Records in U / V files |", "|---|---:|---:|---|"]
    for _, r in per_output.iterrows():
        n = records[records.output == r.output].groupby("uv").size()
        rows.append(f"| {r.output} | {r.truncations:,} | {r.max_cfl:.2f} | {n.get('U', 0)} / {n.get('V', 0)} |")
    mkmd.table(f"MOM6 velocity truncations: {total:,} in {int((per_output.truncations > 0).sum())} of {len(per_output)} output folders", rows)
print(f"{total:,} truncations in total")


### Map of truncation locations

Records are grouped by grid cell (rounded to 0.01°); marker size shows the number of records, colour the maximum
speed. Bathymetry (`deptho`) comes from the static file in the datastore. Land is shown from the model's own mask,
so no map data needs downloading.

In [ ]:
static_path = datastore.search(variable="deptho").df["path"].iloc[0]
static = xr.open_dataset(static_path)
lon2d, lat2d, depth = static["geolon"].values, static["geolat"].values, static["deptho"].values
# geolon/geolat are NaN over processors masked out as land; fill those with the nominal 1-D
# coordinates so pcolormesh can draw (the depth there is NaN anyway, so they show as land).
lon2d = np.where(np.isnan(lon2d), np.broadcast_to(static["xh"].values, lon2d.shape), lon2d)
lat2d = np.where(np.isnan(lat2d), np.broadcast_to(static["yh"].values[:, None], lat2d.shape), lat2d)
step = max(1, lon2d.shape[1] // 1440)  # thin high-resolution grids for plotting only (~0.25 degree)
lon2d, lat2d, depth = lon2d[::step, ::step], lat2d[::step, ::step], depth[::step, ::step]
lon_min = np.nanmin(lon2d)  # tripolar grids run from about -280 to 80; put points in the same range


In [ ]:
def wrap(lon):
    """Longitudes into the grid's range [lon_min, lon_min + 360)."""
    return (np.asarray(lon) - lon_min) % 360 + lon_min


if len(records):
    cells = (records.assign(lon=wrap(records.lon).round(2), lat=records.lat.round(2))
            .groupby(["lon", "lat"])
            .agg(records=("pe", "size"), speed=("speed", "max"), cfl=("cfl", "max"),
                uv=("uv", lambda s: ", ".join(sorted(set(s)))),
                at_cap=("at_cap", "any"))
            .reset_index())
    sizes = 15 + 85 * np.sqrt(cells.records / cells.records.max())

    arctic = cells.lat > 60

    vmax = cells.speed.max()

    fig, axes = plt.subplots(1, 2 if arctic.any() else 1, figsize=(16 if arctic.any() else 10, 5.5),
                             squeeze=False, gridspec_kw={"width_ratios": [2, 1]} if arctic.any() else None)
    panels = [(axes[0, 0], np.ones(len(cells), bool), "Global")]
    if arctic.any():
        panels.append((axes[0, 1], arctic.values, "North of 60°N"))
    for ax, sel, title in panels:
        ax.pcolormesh(lon2d, lat2d, np.where(depth > 0, depth, np.nan), cmap="Blues", shading="auto", vmin=0, vmax=6000)
        ax.set_facecolor("0.8")  # land
        sc = ax.scatter(cells.lon[sel], cells.lat[sel], s=sizes[sel], c=cells.speed[sel], cmap="magma_r",
                        edgecolors="k", linewidths=0.4, vmin=0, vmax=vmax)
        ax.set_title(title)
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")
        if title != "Global":
            ax.set_ylim(60, 90)
            ax.set_xlim(cells.lon[sel].min() - 10, cells.lon[sel].max() + 10)
    fig.colorbar(sc, ax=axes.ravel().tolist(), label="Max truncated speed (m/s)", shrink=0.8)
    fig.suptitle(f"MOM6 velocity truncation records ({len(records)} records at {len(cells)} grid cells)")
else:
    print("No truncation records to map.")


In [ ]:
if len(records):
    mkmd.savefig(
        fig,
        "MOM6 velocity truncation locations",
        f"Grid cells where MOM6 truncated velocities, from the U/V_velocity_truncations files in {len(outputs)} "
        "output folders, over model bathymetry (blue; land grey). Marker size: number of records; colour: maximum "
        "truncated speed. MOM6 writes at most 50 records per PE per run, so the busiest points are undercounted "
        f"(PEs at the cap: {', '.join(map(str, sorted(capped_pes))) or 'none'}).",
        dpi=150,
    )


### Busiest locations

In [ ]:
if len(records):
    top = cells.sort_values("records", ascending=False).head(15)
    rows = ["| Location | Records | Velocity | Max speed (m/s) | Max CFL |", "|---|---:|---|---:|---:|"]
    for _, c in top.iterrows():
        lon = (c.lon + 180) % 360 - 180
        place = (f"[{abs(lon):.2f}°{'E' if lon >= 0 else 'W'}, {abs(c.lat):.2f}°{'N' if c.lat >= 0 else 'S'}]"
                 f"(https://www.google.com/maps/place/{c.lat:.2f},{lon:.2f}/@{c.lat:.2f},{lon:.2f},5z)")  # pin, zoomed out
        rows.append(f"| {place} | {c.records}{'*' if c.at_cap else ''} | {c.uv} | {c.speed:.1f} | {c.cfl:.2f} |")
    mkmd.table("Busiest truncation locations (\\* = a PE at the 50-record cap)", rows)
    top
